In [3]:
import pandas as pd
import numpy as np


In [4]:
playoff_players = pd.read_csv("data/playoff_players.csv")

In [5]:
player_ids = list(zip(playoff_players['PLAYER_ID'].to_list(), playoff_players['TeamID'].to_list()))
display(player_ids[:5])


[(1631105, 1610612765),
 (1630595, 1610612765),
 (1642403, 1610612765),
 (1641842, 1610612765),
 (1630194, 1610612765)]

In [6]:
from nba_api.stats.endpoints import leaguegamelog

logs = leaguegamelog.LeagueGameLog(season='2025-26', season_type_all_star='Playoffs').get_data_frames()[0]
logs

,SEASON_ID,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,WL,MIN,FGM,...,DREB,REB,AST,STL,BLK,TOV,PF,PTS,PLUS_MINUS,VIDEO_AVAILABLE
0,42025,1610612737,ATL,Atlanta Hawks,0042500121,2026-04-18,ATL @ NYK,L,240,38,...,32,40,25,5,3,12,21,102,-11,1
1,42025,1610612739,CLE,Cleveland Cavaliers,0042500131,2026-04-18,CLE vs. TOR,W,240,44,...,26,33,24,9,0,17,28,126,13,1
2,42025,1610612743,DEN,Denver Nuggets,0042500161,2026-04-18,DEN vs. MIN,W,240,38,...,39,47,27,7,1,14,17,116,11,1
3,42025,1610612745,HOU,Houston Rockets,0042500171,2026-04-18,HOU @ LAL,L,240,35,...,23,44,24,13,3,13,22,98,-9,1
4,42025,1610612747,LAL,Los Angeles Lakers,0042500171,2026-04-18,LAL vs. HOU,W,240,40,...,32,35,29,7,8,20,25,107,9,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
127,42025,1610612747,LAL,Los Angeles Lakers,0042500224,2026-05-11,LAL vs. OKC,L,240,38,...,30,37,18,3,5,19,22,110,-5,1
128,42025,1610612759,SAS,San Antonio Spurs,0042500235,2026-05-12,SAS vs. MIN,W,240,47,...,39,50,25,9,8,15,24,126,29,1
129,42025,1610612750,MIN,Minnesota Timberwolves,0042500235,2026-05-12,MIN @ SAS,L,240,32,...,32,42,17,7,4,16,24,97,-29,1
130,42025,1610612765,DET,Detroit Pistons,0042500205,2026-05-13,DET vs. CLE,L,265,42,...,25,40,28,10,10,15,26,113,-4,1


In [7]:
# currentLogs = pd.read_csv("data/playoff_games.csv")
# logs = logs.astype({"SEASON_ID": "int64", "GAME_ID": "int64"})
# diff = logs.compare(currentLogs)
# diff

In [8]:
def convert_game_id_list_into_str(game_id_list):
  game_id_format = ""
  for idx, game_id in enumerate(game_id_list):
    if idx == len(game_id_list) - 1:
      game_id_format += str(game_id)
    else:
      game_id_format += str(game_id) + "|"

  return game_id_format

In [9]:
logs.to_csv('data/playoff_games.csv', index=False, encoding='utf-8')

In [10]:
player_games = []
for idx, (player_id, team_id) in enumerate(player_ids):
  game_ids = logs.query(f'TEAM_ID == {team_id}')['GAME_ID'].to_list()
  game_id_format =""
  for idx, game_id in enumerate(game_ids):
    if idx == len(game_ids) - 1:
      game_id_format += str(game_id)
    else:
      game_id_format += str(game_id) + "|"
  player_games.append([player_id, game_id_format])
player_games

[[1631105,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042500202|0042500203|0042500204|0042500205'],
 [1630595,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042500202|0042500203|0042500204|0042500205'],
 [1642403,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042500202|0042500203|0042500204|0042500205'],
 [1641842,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042500202|0042500203|0042500204|0042500205'],
 [1630194,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042500202|0042500203|0042500204|0042500205'],
 [1627747,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042500202|0042500203|0042500204|0042500205'],
 [1641709,
  '0042500101|0042500102|0042500103|0042500104|0042500105|0042500106|0042500107|0042500201|0042

In [11]:
import time
import random
import requests
from nba_api.stats.endpoints import cumestatsplayer

def safe_cume_stats(player_id, game_ids, retries=6, timeout=10):
    delay = 1

    for attempt in range(1, retries + 1):
        try:
            return cumestatsplayer.CumeStatsPlayer(
                player_id=player_id,
                game_ids=game_ids,
                timeout=timeout
            )

        except (TimeoutError, requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError, KeyError) as e:
            print(f"Attempt {attempt} failed for player_id {player_id} with game_ids {game_ids}: {e}")

            if attempt == retries:
                raise

            time.sleep(delay + random.uniform(0, 0.5))
            delay *= 2.4

        except Exception as e:
            print(f"Non-retryable error for player_id {player_id} with game_ids {game_ids}: {e}")
            raise


In [12]:
import os
import pickle

# Key is player_id
def load_player_cache(key):
  path = f"cache/{key}.pkl"
  if os.path.exists(path):
    return pickle.load(open(path, "rb"))
  return {"processed_game_ids": [], "game_by_game": None, "cumulative": None}

# key is player_id
def save_player_cache(key, value):
  os.makedirs("cache", exist_ok=True)
  pickle.dump(value, open(f"cache/{key}.pkl", "wb"))

In [13]:
def convert_game_id_str_into_list(game_id_str):
  return game_id_str.split("|")
convert_game_id_str_into_list(player_games[1][1])

['0042500101',
 '0042500102',
 '0042500103',
 '0042500104',
 '0042500105',
 '0042500106',
 '0042500107',
 '0042500201',
 '0042500202',
 '0042500203',
 '0042500204',
 '0042500205']

In [14]:
ADDITIVE_COLS = [
    "GP", "GS",
    "ACTUAL_MINUTES", "ACTUAL_SECONDS",
    "FG", "FGA", "FG3", "FG3A", "FT", "FTA",
    "OFF_REB", "DEF_REB", "TOT_REB",
    "AST", "PF", "DQ", "STL", "TURNOVERS", "BLK", "PTS",
]

MAX_COLS = [
    "MAX_ACTUAL_MINUTES", "MAX_ACTUAL_SECONDS",
    "MAX_REB", "MAX_AST", "MAX_STL", "MAX_TURNOVERS", "MAX_BLK", "MAX_PTS"
]

PCT_COLS = ["FG_PCT", "FG3_PCT", "FT_PCT"]
AVG_COLS = ["AVG_ACTUAL_MINUTES", "AVG_ACTUAL_SECONDS", "AVG_TOT_REB",
              "AVG_AST", "AVG_STL", "AVG_TURNOVERS", "AVG_BLK", "AVG_PTS"]
PER_MIN_COLS = ["PER_MIN_TOT_REB", "PER_MIN_AST", "PER_MIN_STL",
                "PER_MIN_TURNOVERS", "PER_MIN_BLK", "PER_MIN_PTS"]

def recalculate_derived_cols(df):
    total_minutes = df["ACTUAL_MINUTES"] + df["ACTUAL_SECONDS"] / 60

    # Percentages
    df["FG_PCT"]  = df["FG"]  / df["FGA"].replace(0, pd.NA)
    df["FG3_PCT"] = df["FG3"] / df["FG3A"].replace(0, pd.NA)
    df["FT_PCT"]  = df["FT"]  / df["FTA"].replace(0, pd.NA)

    # Averages per game
    gp = df["GP"].replace(0, pd.NA)
    df["AVG_ACTUAL_MINUTES"]  = df["ACTUAL_MINUTES"] / gp
    df["AVG_ACTUAL_SECONDS"]  = df["ACTUAL_SECONDS"] / gp
    df["AVG_TOT_REB"]         = df["TOT_REB"]   / gp
    df["AVG_AST"]             = df["AST"]        / gp
    df["AVG_STL"]             = df["STL"]        / gp
    df["AVG_TURNOVERS"]       = df["TURNOVERS"]  / gp
    df["AVG_BLK"]             = df["BLK"]        / gp
    df["AVG_PTS"]             = df["PTS"]        / gp

    # Per-minute
    tm = total_minutes.replace(0, pd.NA)
    df["PER_MIN_TOT_REB"]   = df["TOT_REB"]  / tm
    df["PER_MIN_AST"]       = df["AST"]       / tm
    df["PER_MIN_STL"]       = df["STL"]       / tm
    df["PER_MIN_TURNOVERS"] = df["TURNOVERS"] / tm
    df["PER_MIN_BLK"]       = df["BLK"]       / tm
    df["PER_MIN_PTS"]       = df["PTS"]       / tm

    return df




In [15]:
def fetch_incremental_cume(player_id, all_game_ids_list):
  cache = load_player_cache(player_id)

  # Convert processed_game_ids in cache to set for faster lookup
  processed_game_ids_set = set(cache["processed_game_ids"])

  # Identify new games by comparing with cached games
  new_games_list = [gid for gid in all_game_ids_list if gid not in processed_game_ids_set]

  # Nothing to fetch, already updated
  if not new_games_list:
    return cache["game_by_game"], cache["cumulative"]

  # Join new_games_list into a pipe-separated string for the API call
  game_ids_api_format = convert_game_id_list_into_str(new_games_list)

  # Fetch new data

  stats = safe_cume_stats(player_id, game_ids_api_format)
  new_game_by_game = stats.get_data_frames()[0]
  new_cumulative = stats.get_data_frames()[1]

  cache["processed_game_ids"].extend(new_games_list)

  if new_game_by_game.empty or new_cumulative.empty:
    save_player_cache(player_id, cache)
    return cache["game_by_game"], cache["cumulative"]
  
  new_game_by_game["GAME_ID"] = new_games_list

  if cache["game_by_game"] is None:
    merged_gbg = new_game_by_game
  else:
    merged_gbg = pd.concat([cache["game_by_game"], new_game_by_game], ignore_index=True)

  if cache["cumulative"] is None:
    merged_cumulative = new_cumulative
  else:
    merged_cumulative = cache["cumulative"].copy()
    merged_cumulative[ADDITIVE_COLS] += new_cumulative[ADDITIVE_COLS].values
    merged_cumulative[MAX_COLS] = np.maximum(merged_cumulative[MAX_COLS].values, new_cumulative[MAX_COLS].values)
    merged_cumulative = recalculate_derived_cols(merged_cumulative)

  cache["game_by_game"] = merged_gbg
  cache["cumulative"] = merged_cumulative

  save_player_cache(player_id, cache)

  return merged_gbg, merged_cumulative

In [17]:
all_gbg = []
all_totals = []

for player_id, game_ids in player_games:
  gbg, totals = fetch_incremental_cume(player_id, convert_game_id_str_into_list(game_ids))
  gbg["PLAYER_ID"] = player_id
  totals.rename(columns={"PERSON_ID": "PLAYER_ID"}, inplace=True)
  all_gbg.append(gbg)
  all_totals.append(totals)

df_game_by_game = pd.concat(all_gbg, ignore_index=True)
df_total = pd.concat(all_totals, ignore_index=True)

In [18]:
def fix_seconds_overflow(df):
    overflow_minutes = df["ACTUAL_SECONDS"] // 60
    df["ACTUAL_MINUTES"] += overflow_minutes
    df["ACTUAL_SECONDS"] = df["ACTUAL_SECONDS"] % 60
    return df

df_total = fix_seconds_overflow(df_total)

In [19]:
df_total


,DISPLAY_FI_LAST,PLAYER_ID,JERSEY_NUM,GP,GS,ACTUAL_MINUTES,ACTUAL_SECONDS,FG,FGA,FG_PCT,...,AVG_STL,AVG_TURNOVERS,AVG_BLK,AVG_PTS,PER_MIN_TOT_REB,PER_MIN_AST,PER_MIN_STL,PER_MIN_TURNOVERS,PER_MIN_BLK,PER_MIN_PTS
0,J. Duren,1631105,0,12,12,369,8,46,92,0.5,...,0.666667,2.416667,1.0,10.083333,0.268196,0.073144,0.021672,0.078562,0.032509,0.327795
1,C. Cunningham,1630595,2,12,12,492,47,115,259,0.444015,...,1.083333,5.75,0.666667,30.0,0.133933,0.186695,0.026381,0.140021,0.016234,0.730544
2,R. Holland II,1641842,5,7,0,49,58,3,10,0.3,...,0.57,0.4,0.29,1.4,11.5,1.0,3.84,2.9,1.92,9.6
3,P. Reed,1630194,7,7,0,61,25,19,28,0.678571,...,0.142857,0.571429,0.714286,6.571429,0.423338,0.048847,0.016282,0.065129,0.081411,0.748982
4,C. LeVert,1627747,8,11,0,170,21,23,53,0.433962,...,0.545455,0.454545,0.454545,5.272727,0.140886,0.064573,0.035222,0.029351,0.029351,0.340475
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
225,T. Eason,1631106,17,6,4,195,8,31,65,0.477,...,2.5,0.8,0.67,13.8,9.8,2.5,3.69,1.2,0.98,20.4
226,J. Okogie,1629006,20,6,2,104,26,11,25,0.44,...,1.17,0.5,0.0,4.8,6.4,2.3,3.22,1.4,0.0,13.3
227,A. Sengun,1630578,28,6,6,231,57,46,99,0.465,...,1.83,3.0,1.33,20.3,12.6,5.8,2.28,3.7,1.66,25.2
228,C. Capela,203991,30,4,0,21,47,3,8,0.375,...,0.0,0.3,0.25,1.5,15.4,2.2,0.0,2.2,2.2,13.2


In [21]:
df_game_by_game['DATE_STRING'] = df_game_by_game["DATE_EST"].str.replace("/", "")
df_game_by_game

,DATE_EST,VISITOR_TEAM,HOME_TEAM,GP,GS,ACTUAL_MINUTES,ACTUAL_SECONDS,FG,FGA,FG_PCT,...,AST,PF,DQ,STL,TURNOVERS,BLK,PTS,AVG_PTS,PLAYER_ID,DATE_STRING
0,04/19/2026,Magic,Pistons,1,1,32,48,3,4,0.75,...,1,3,0,0,3,1,8,8.0,1631105,04192026
1,04/22/2026,Magic,Pistons,1,1,31,41,4,10,0.4,...,4,1,0,1,3,0,11,9.5,1631105,04222026
2,04/25/2026,Pistons,Magic,1,1,27,27,3,10,0.3,...,1,6,1,0,2,5,8,9.0,1631105,04252026
3,04/27/2026,Pistons,Magic,1,1,30,45,5,8,0.625,...,3,5,0,1,4,1,12,9.8,1631105,04272026
4,04/29/2026,Magic,Pistons,1,1,27,34,4,6,0.667,...,2,5,0,0,3,2,12,10.2,1631105,04292026
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1475,04/21/2026,Rockets,Lakers,1,0,4,40,0,3,0.0,...,0,0,0,0,0,1,0,1.0,203991,04212026
1476,04/24/2026,Lakers,Rockets,1,0,3,40,1,2,0.5,...,1,1,0,0,0,0,2,1.3,203991,04242026
1477,05/01/2026,Lakers,Rockets,1,0,2,7,1,1,1.0,...,0,0,0,0,0,0,2,1.5,203991,05012026
1478,04/24/2026,Lakers,Rockets,1,0,7,31,0,2,0.0,...,0,1,0,0,1,0,0,0.0,201145,04242026


In [23]:
df_game_by_game.to_csv('data/game_by_game.csv', index=False, encoding='utf-8')
df_total.to_csv('data/totals.csv', index=False, encoding='utf-8')

In [29]:
# Update Postgres DB
from sqlalchemy import create_engine
from config import Config

engine = create_engine(Config.SQLALCHEMY_DATABASE_URI)

In [25]:
logs.columns = logs.columns.str.lower()
df_game_by_game.columns = df_game_by_game.columns.str.lower()
df_total.columns = df_total.columns.str.lower()

In [26]:
from sqlalchemy import text

logs.to_sql("games", engine, schema="nba_data", if_exists="replace", index=True, index_label="id")
with engine.connect() as conn:
    conn.execute(text("ALTER TABLE nba_data.games ADD PRIMARY KEY (id);"))
    conn.commit()

In [27]:
df_game_by_game.to_sql("game_by_game", engine, schema="nba_data", if_exists="replace", index=True, index_label="id")
with engine.connect() as conn:
    conn.execute(text('ALTER TABLE nba_data.game_by_game ADD PRIMARY KEY (id);'))
    conn.commit()

In [28]:
df_total.to_sql("totals", engine, schema="nba_data", if_exists="replace", index=False)
with engine.connect() as conn:
    conn.execute(text('ALTER TABLE nba_data.totals ADD PRIMARY KEY (player_id);'))
    conn.commit()